# 掼蛋·扑克牌识别模型训练（Colab 一键版）

在 **Google Colab 免费 GPU** 上训练识别扑克牌「数字+花色」的模型(YOLO)。

**用法**：菜单「运行时 → 更改运行时类型 → 选 GPU」，再「运行时 → 全部运行」。


## 1. 确认已分到 GPU


In [ ]:
!nvidia-smi

## 2. 安装训练框架（Ultralytics YOLO）


In [ ]:
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()

## 3. 获取扑克牌数据集（自动挑可用版本）

只需把 `api_key` 换成你 Roboflow 账号的 Private API Key；
代码会自动列出该数据集所有版本、逐个尝试，找到带 YOLOv8 导出(data.yaml)的就用。


In [ ]:
import shutil, os, glob, zipfile
from roboflow import Roboflow

rf = Roboflow(api_key="把你的Private_API_Key粘到这里")
project = rf.workspace("augmented-startups").project("playing-cards-ow27d")

try:
    vers = [v.version for v in project.versions()]
except Exception:
    vers = [1, 2, 3, 4, 5, 6]
print('要尝试的版本:', vers)

for d in glob.glob('/content/Playing-Cards-*') + glob.glob('/content/playing-cards-*'):
    shutil.rmtree(d, ignore_errors=True)

dataset = None
for vn in vers:
    try:
        ds = project.version(vn).download("yolov8")
    except Exception as e:
        print(f'版本{vn}: 跳过（{type(e).__name__}）'); continue
    loc = ds.location
    if not glob.glob(os.path.join(loc, '**/data.yaml'), recursive=True):
        for z in glob.glob(os.path.join(loc, '*.zip')):
            try:
                with zipfile.ZipFile(z) as zf: zf.extractall(loc)
            except Exception: pass
    if glob.glob(os.path.join(loc, '**/data.yaml'), recursive=True):
        dataset = ds; print(f'✅ 版本{vn} 可用 -> {loc}'); break
    print(f'版本{vn}: 无 YOLOv8 导出，换下一个')

assert dataset, '所有版本都没有 YOLOv8 导出——需在数据集网页 Generate 一个 YOLOv8 版本'
print('最终目录内容:', os.listdir(dataset.location))

## 4. 开始训练（自动用上一步的数据集）


In [ ]:
import glob, os
from ultralytics import YOLO
DATA = glob.glob(os.path.join(dataset.location, '**/data.yaml'), recursive=True)[0]
print('使用数据集配置:', DATA)
model = YOLO('yolov8n.pt')   # 要更准可换 yolov8s.pt / yolov8m.pt
model.train(data=DATA, epochs=50, imgsz=640, batch=16,
            patience=20, project='guandan_cards', name='exp')

## 5. 看效果（验证集指标）


In [ ]:
metrics = model.val()
print('mAP50:', metrics.box.map50, ' mAP50-95:', metrics.box.map)

## 6. 导出模型（给 App 用）


In [ ]:
best = 'guandan_cards/exp/weights/best.pt'
m = YOLO(best)
m.export(format='onnx')
print('模型：', best, ' 和 同目录 best.onnx')

## 7. 下载模型到本地


In [ ]:
from google.colab import files
files.download('guandan_cards/exp/weights/best.pt')
files.download('guandan_cards/exp/weights/best.onnx')

## 8. 拿回模型后怎么用

把 `best.pt` 放到本仓库 `vision/`，运行 `python vision/recognize.py 照片.jpg`，
会输出识别到的牌并调用掼蛋引擎给『自动组牌 + 出牌建议』。手机/眼镜端用 `best.onnx`。
